# Critical Passage Relabeling Pipeline

This notebook implements **Enhancement 4: Critical Passage Relabeling** from the writeup.

## Prerequisites

**Run `kd_pipeline_final.ipynb` first.** This notebook consumes its outputs:

- **Detection files** (`artifacts/curation/`): `relabel_crit_ge4.jsonl`, `relabel_crit_ge3.jsonl`, `relabel_cons_topk.jsonl`
- **Reranker outputs** (`artifacts/rerank/`): Voyage, BGE-m3, and BGE-Gemma2 rankings per corpus

## What This Notebook Does

1. **Build LLM Input Packages** - Converts detection output into packages with calibration anchors
2. **LLM Labeling** - Runs GPT-5 and Gemini to label critical passages
3. **Compare & Combine** - Precision-first fusion of both models' labels
4. **Merge to Training** - Updates training data with new labels

## Step 0: Configuration

In [ ]:
import os
import json
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from collections import defaultdict

# ==================== PATHS ====================
# Input paths
TRAIN_JSONL = os.getenv("TRAIN_JSONL", "/content/hsrc_train.jsonl")
CORPUS_JSONL = os.getenv("CORPUS_JSONL", "/content/hsrc_corpus.jsonl")
CURATION_DIR = os.getenv("CURATION_DIR", "/content/artifacts/curation")
RERANK_DIR = os.getenv("RERANK_DIR", "/content/artifacts/rerank")

# Output paths
OUTPUT_DIR = os.getenv("OUTPUT_DIR", "/content/crit_relabel")
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Detection input files (from kd_pipeline_final.ipynb)
DETECTION_FILES = {
    "crit_ge4": os.path.join(CURATION_DIR, "relabel_crit_ge4.jsonl"),
    "crit_ge3": os.path.join(CURATION_DIR, "relabel_crit_ge3.jsonl"),
    "zero_pos": os.path.join(CURATION_DIR, "relabel_zero_pos.jsonl"),
    "cons_topk": os.path.join(CURATION_DIR, "relabel_cons_topk.jsonl"),
}

# ==================== SETTINGS ====================
PASSAGE_MAX_TOKENS = int(os.getenv("PASSAGE_MAX_TOKENS", "600"))
STRICT_REQUIRE_ALL_ANCHORS = False  # If True, skip queries missing any label 0-4
FUSE_WEIGHTS = {"voyage": 0.50, "bge_m3": 0.25, "bge_gemma": 0.25}
RRF_K = 60

# Token truncation setup
try:
    import tiktoken
    _enc = tiktoken.get_encoding(os.getenv("TIKTOKEN_ENCODING", "o200k_base"))
    def truncate_to_tokens(text: str, max_tokens: int = PASSAGE_MAX_TOKENS) -> str:
        toks = _enc.encode(text)
        if len(toks) <= max_tokens:
            return text
        return _enc.decode(toks[:max_tokens]) + "…"
    TOKENIZER_INFO = f"tiktoken:{_enc.name}"
except ImportError:
    def truncate_to_tokens(text: str, max_tokens: int = PASSAGE_MAX_TOKENS) -> str:
        words = text.split()
        if len(words) <= max_tokens:
            return text
        return " ".join(words[:max_tokens]) + "…"
    TOKENIZER_INFO = "whitespace-fallback"

print(f"[CONFIG] TRAIN_JSONL: {TRAIN_JSONL}")
print(f"[CONFIG] CORPUS_JSONL: {CORPUS_JSONL}")
print(f"[CONFIG] CURATION_DIR: {CURATION_DIR}")
print(f"[CONFIG] OUTPUT_DIR: {OUTPUT_DIR}")
print(f"[CONFIG] PASSAGE_MAX_TOKENS: {PASSAGE_MAX_TOKENS}")
print(f"[CONFIG] Tokenizer: {TOKENIZER_INFO}")

## Step 1: Load Data

Load training data (queries + labels), corpus (passage texts), and fused rankings.

In [ ]:
# ==================== DATA LOADERS ====================

def iter_jsonl(path: str):
    """Iterate over JSONL file."""
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                yield json.loads(s)

def load_corpus(corpus_path: str) -> Dict[str, str]:
    """Load corpus: doc_id -> passage text."""
    corpus = {}
    for row in iter_jsonl(corpus_path):
        doc_id = row.get("uuid") or row.get("paragraph_uuid") or row.get("doc_id")
        text = row.get("passage") or row.get("text") or ""
        if doc_id:
            corpus[doc_id] = text
    print(f"[LOAD] Corpus: {len(corpus):,} passages")
    return corpus

def load_train_data(train_path: str) -> Tuple[Dict, Dict, Dict, Dict]:
    """
    Load training data with structures needed for anchor building.
    
    Returns:
        - queries: qid -> {"text": str, "corpus": str}
        - docs_by_label: qid -> {label: [doc_id, ...]} - docs grouped by label per query
        - labeled_passages: qid -> {doc_id: passage_text} - passage text for labeled docs
        - label4_docs: qid -> set(doc_id) - docs with label=4 for quick lookup
    """
    queries = {}
    docs_by_label = defaultdict(lambda: defaultdict(list))
    labeled_passages = defaultdict(dict)
    label4_docs = defaultdict(set)
    
    for row in iter_jsonl(train_path):
        qid = row.get("query_uuid")
        if not qid:
            continue
        
        queries[qid] = {
            "text": row.get("query", ""),
            "corpus": row.get("corpus", "unknown")
        }
        
        # Build paragraph index: idx -> {uuid, passage}
        paragraphs = row.get("paragraphs", {})
        pmap = {}
        for key, para_info in paragraphs.items():
            idx = key.replace("paragraph_", "")
            pmap[idx] = {
                "uuid": para_info.get("uuid") or para_info.get("paragraph_uuid"),
                "passage": para_info.get("passage", "")
            }
        
        # Extract labels and build structures
        target_actions = row.get("target_actions", {})
        for key, label_val in target_actions.items():
            idx = key.replace("target_action_", "")
            if idx not in pmap:
                continue
            
            try:
                label = int(str(label_val).strip())
            except (ValueError, TypeError):
                continue
            
            doc_id = pmap[idx]["uuid"]
            passage = pmap[idx]["passage"]
            
            if doc_id:
                docs_by_label[qid][label].append(doc_id)
                labeled_passages[qid][doc_id] = passage
                if label == 4:
                    label4_docs[qid].add(doc_id)
    
    print(f"[LOAD] Training data: {len(queries):,} queries")
    return queries, dict(docs_by_label), dict(labeled_passages), dict(label4_docs)

def load_rerank_map(rerank_path: str) -> Dict[str, Dict]:
    """Load reranker output: qid -> {corpus, docs: [doc_id, ...]}."""
    result = {}
    if not Path(rerank_path).exists():
        return result
    for row in iter_jsonl(rerank_path):
        qid = row.get("query_uuid")
        if qid:
            result[qid] = {
                "corpus": row.get("corpus", ""),
                "docs": [d.get("doc_id") or d.get("uuid") for d in row.get("results", [])]
            }
    return result

def load_fused_rankings(rerank_dir: str, corpora: List[str] = ["kz", "wiki", "knesset"]) -> Dict[str, Dict]:
    """
    Load and fuse rankings from multiple rerankers.
    Returns: qid -> {"rank": {doc_id: rank}, "docs": [doc_id, ...]}
    """
    voyage_all, bge_m3_all, bge_gemma_all = {}, {}, {}
    
    for corpus in corpora:
        voyage_path = Path(rerank_dir) / f"{corpus}_rerank_voyage.jsonl"
        bge_m3_path = Path(rerank_dir) / f"{corpus}_rerank_bge_m3.jsonl"
        bge_gemma_path = Path(rerank_dir) / f"{corpus}_rerank_bge_gemma2.jsonl"
        
        voyage_all.update(load_rerank_map(str(voyage_path)))
        bge_m3_all.update(load_rerank_map(str(bge_m3_path)))
        bge_gemma_all.update(load_rerank_map(str(bge_gemma_path)))
    
    print(f"[LOAD] Rerank maps: voyage={len(voyage_all)}, bge_m3={len(bge_m3_all)}, bge_gemma={len(bge_gemma_all)}")
    
    # Fuse rankings using RRF
    fused = {}
    all_qids = set(voyage_all) | set(bge_m3_all) | set(bge_gemma_all)
    
    for qid in all_qids:
        sources = {
            "voyage": voyage_all.get(qid, {}).get("docs", []),
            "bge_m3": bge_m3_all.get(qid, {}).get("docs", []),
            "bge_gemma": bge_gemma_all.get(qid, {}).get("docs", [])
        }
        docs_list = rrf_fuse(sources, FUSE_WEIGHTS, k=100, rrf_k=RRF_K)
        # Build rank map: doc_id -> rank (0-indexed)
        rank_map = {doc_id: rank for rank, doc_id in enumerate(docs_list)}
        fused[qid] = {"rank": rank_map, "docs": docs_list}
    
    print(f"[LOAD] Fused rankings: {len(fused):,} queries")
    return fused

def rrf_fuse(sources: Dict[str, List[str]], weights: Dict[str, float], k: int = 100, rrf_k: int = 60) -> List[str]:
    """Reciprocal Rank Fusion."""
    scores = defaultdict(float)
    for name, docs in sources.items():
        w = weights.get(name, 1.0)
        for rank, doc_id in enumerate(docs, start=1):
            scores[doc_id] += w / (rrf_k + rank)
    
    sorted_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, _ in sorted_docs[:k]]

In [ ]:
# Load all data
corpus = load_corpus(CORPUS_JSONL)
queries, docs_by_label, labeled_passages, label4_docs = load_train_data(TRAIN_JSONL)
fused_rankings = load_fused_rankings(RERANK_DIR)

## Step 2: Build LLM Input Packages

For each detected critical passage, build an LLM input package with:

- `query`: {uuid, text}
- `anchors`: **One passage per label (0-4)** from the same query - used as calibration examples
- `candidate`: The unlabeled passage to label
- `ref4`: A label-4 passage ranked **below** the candidate (preferred) or best label-4 available

### Anchor Selection Logic

For each label value (0, 1, 2, 3, 4) that exists for this query:
1. Find all docs with that label
2. Pick the one with best fused rank
3. Truncate passage to 600 tokens

Result: Up to 5 anchors (one per label), showing the LLM what each relevance level looks like for this specific query.

In [ ]:
# ==================== PACKAGE BUILDER ====================

def pick_best_by_rank(doc_ids: List[str], rank_map: Dict[str, int]) -> Optional[str]:
    """Pick the doc with best (lowest) fused rank. Returns None if none are ranked."""
    ranked = [(d, rank_map[d]) for d in doc_ids if d in rank_map]
    if not ranked:
        return doc_ids[0] if doc_ids else None  # fallback to first
    return min(ranked, key=lambda x: x[1])[0]

def build_anchors(
    qid: str,
    docs_by_label: Dict[str, Dict[int, List[str]]],
    labeled_passages: Dict[str, Dict[str, str]],
    corpus: Dict[str, str],
    rank_map: Dict[str, int]
) -> List[Dict]:
    """
    Build anchors: one passage per label (0-4) from the same query.
    Picks best-ranked doc for each label, truncates to PASSAGE_MAX_TOKENS.
    """
    labels_for_q = docs_by_label.get(qid, {})
    anchors = []
    
    for label in (0, 1, 2, 3, 4):
        doc_ids = labels_for_q.get(label, [])
        if not doc_ids:
            continue
        
        # Pick best by fused rank
        best_doc = pick_best_by_rank(doc_ids, rank_map)
        if not best_doc:
            continue
        
        # Get passage text (prefer labeled_passages, fallback to corpus)
        passage = labeled_passages.get(qid, {}).get(best_doc, "") or corpus.get(best_doc, "")
        if passage:
            anchors.append({
                "label": label,
                "passage": truncate_to_tokens(passage, PASSAGE_MAX_TOKENS)
            })
    
    return anchors

def find_ref4(
    qid: str,
    candidate_doc_id: str,
    candidate_rank: int,
    label4_docs: Dict[str, set],
    labeled_passages: Dict[str, Dict[str, str]],
    corpus: Dict[str, str],
    rank_map: Dict[str, int]
) -> Optional[Dict]:
    """
    Find ref4: a label-4 passage ranked BELOW the candidate.
    Fallback: best-ranked label-4 overall.
    """
    l4_set = label4_docs.get(qid, set())
    if not l4_set:
        return None
    
    # Sort label-4 docs by fused rank
    l4_ranked = sorted(
        [(d, rank_map[d]) for d in l4_set if d in rank_map],
        key=lambda x: x[1]
    )
    
    # Prefer a label-4 ranked BELOW the candidate
    ref4_doc = None
    for doc_id, rank in l4_ranked:
        if rank > candidate_rank:
            ref4_doc = doc_id
            break
    
    # Fallback: best-ranked label-4
    if ref4_doc is None and l4_ranked:
        ref4_doc = l4_ranked[0][0]
    
    if ref4_doc is None:
        return None
    
    # Get passage
    passage = labeled_passages.get(qid, {}).get(ref4_doc, "") or corpus.get(ref4_doc, "")
    if not passage:
        return None
    
    return {
        "doc_id": ref4_doc,
        "passage": truncate_to_tokens(passage, PASSAGE_MAX_TOKENS)
    }

def build_llm_input_packages(
    detection_file: str,
    queries: Dict[str, Dict],
    docs_by_label: Dict[str, Dict[int, List[str]]],
    labeled_passages: Dict[str, Dict[str, str]],
    label4_docs: Dict[str, set],
    corpus: Dict[str, str],
    fused_rankings: Dict[str, Dict],
    output_file: str
) -> int:
    """
    Build LLM input packages from detection output.
    
    For each (qid, doc_id) in detection file:
    1. Build anchors: one passage per label (0-4) from same query
    2. Get candidate passage
    3. Find ref4: label-4 passage ranked below candidate (preferred)
    
    Returns: Number of packages written
    """
    if not Path(detection_file).exists():
        print(f"[WARN] Detection file not found: {detection_file}")
        return 0
    
    written = 0
    skips = {"no_query": 0, "no_fused": 0, "no_candidate": 0, "no_ref4": 0, "no_anchors": 0}
    
    with open(output_file, "w", encoding="utf-8") as wf:
        for row in iter_jsonl(detection_file):
            qid = row.get("query_uuid")
            doc_id = row.get("doc_id")
            
            if not qid or not doc_id:
                continue
            
            # Get query info
            q_info = queries.get(qid)
            if not q_info:
                skips["no_query"] += 1
                continue
            
            # Get fused ranking for this query
            fused_entry = fused_rankings.get(qid)
            if not fused_entry:
                skips["no_fused"] += 1
                continue
            
            rank_map = fused_entry["rank"]
            candidate_rank = rank_map.get(doc_id)
            if candidate_rank is None:
                skips["no_fused"] += 1
                continue
            
            # Get candidate passage
            candidate_passage = corpus.get(doc_id, "")
            if not candidate_passage:
                skips["no_candidate"] += 1
                continue
            
            # Build anchors (one per label 0-4)
            anchors = build_anchors(qid, docs_by_label, labeled_passages, corpus, rank_map)
            
            if STRICT_REQUIRE_ALL_ANCHORS and len(anchors) < 5:
                skips["no_anchors"] += 1
                continue
            
            if not anchors:
                skips["no_anchors"] += 1
                continue
            
            # Find ref4 (label-4 ranked below candidate, or best label-4)
            ref4 = find_ref4(qid, doc_id, candidate_rank, label4_docs, labeled_passages, corpus, rank_map)
            if ref4 is None:
                skips["no_ref4"] += 1
                continue
            
            # Build package
            package = {
                "query": {"uuid": qid, "text": q_info["text"]},
                "anchors": anchors,
                "candidate": {
                    "doc_id": doc_id,
                    "passage": truncate_to_tokens(candidate_passage, PASSAGE_MAX_TOKENS)
                },
                "ref4": ref4
            }
            
            wf.write(json.dumps(package, ensure_ascii=False) + "\n")
            written += 1
    
    print(f"[BUILD] {Path(detection_file).name} -> {Path(output_file).name}")
    print(f"        Written: {written:,} | Skipped: {skips}")
    
    return written

In [ ]:
# Build LLM input packages for each detection type

for detection_type, detection_path in DETECTION_FILES.items():
    output_path = os.path.join(OUTPUT_DIR, f"llm_inputs_relabel_{detection_type}.jsonl")
    build_llm_input_packages(
        detection_file=detection_path,
        queries=queries,
        docs_by_label=docs_by_label,
        labeled_passages=labeled_passages,
        label4_docs=label4_docs,
        corpus=corpus,
        fused_rankings=fused_rankings,
        output_file=output_path
    )

print("\n[DONE] All LLM input packages built.")

In [ ]:
# Preview a package
preview_file = os.path.join(OUTPUT_DIR, "llm_inputs_relabel_crit_ge4.jsonl")
if Path(preview_file).exists():
    print("=" * 80)
    print("PREVIEW: llm_inputs_relabel_crit_ge4.jsonl")
    print("=" * 80)
    with open(preview_file, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 1:
                break
            pkg = json.loads(line)
            print(f"Query: {pkg['query']['text'][:100]}...")
            print(f"\nAnchors ({len(pkg['anchors'])} labels):")
            for a in pkg['anchors']:
                print(f"  L={a['label']}: {a['passage'][:80]}...")
            print(f"\nCandidate [{pkg['candidate']['doc_id'][:20]}...]:")
            print(f"  {pkg['candidate']['passage'][:120]}...")
            print(f"\nRef4 [{pkg['ref4']['doc_id'][:20]}...]:")
            print(f"  {pkg['ref4']['passage'][:120]}...")

## Step 3: LLM Labeling

Run GPT-5 and Gemini labeling on the packages. Both models use the same system prompt with:
- Hebrew & Israeli context awareness
- Calibration via same-query anchors  
- 0-4 relevance scale with confidence scores

**Models:**
- GPT-5 (`gpt-5`) via OpenAI API with reasoning
- Gemini 2.5 Pro (`gemini-2.5-pro`) via Google GenAI with thinking

**Features:**
- Resume-safe: skips already-labeled pairs
- Async with semaphore-controlled concurrency
- Robust JSON extraction from model output

In [ ]:
# ==================== GPT-5 LABELING ====================
# Aligned with final_relabel/new.py

import asyncio
import re
import sys
from openai import AsyncOpenAI

# Config
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")  # Set your key
MODEL_GPT = os.getenv("MODEL_GPT", "gpt-5")
CONCURRENCY_GPT = int(os.getenv("CONCURRENCY_GPT", "6"))
MAX_RETRY = int(os.getenv("MAX_RETRY", "3"))
DEBUG = bool(int(os.getenv("DEBUG", "0")))

SYSTEM_PROMPT_GPT = """Rate the relevance of a passage to a query on a 0–4 scale using a single JSON object you will receive.

Input JSON fields:
- query: { "uuid", "text" }
- anchors: list of examples from the same query, each { "label": 0..4, "passage": "…" } (use these to calibrate the scale)
- candidate: { "doc_id", "passage" } (the passage to score)
- ref4: { "doc_id", "passage" } (an additional reference passage from the same query that ranked just below the candidate using a reranker)

Relevance scale (0–4)
0 – No substantive relation to the query.
1 – Weak/general relation; no direct answer.
2 – Partially relevant; some matching info but key elements are missing.
3 – Strong relevance; mostly answers the query with minor gaps.
4 – Fully relevant; a direct and complete answer.

Hebrew & Israel cues
- Texts are in Hebrew (RTL) and may lack niqqud; treat common prefixes/suffixes (ב/כ/ל/מ/ו/ה-, -ים/-ות) and spelling variants as equivalent.
- Prefer content anchored in the Israeli context when its relevant to the query (e.g., laws/procedures/institutions like ביטוח לאומי, שירות התעסוקה, רשות המסים, משרד הפנים, קופות חולים). If the passage is about a non-Israeli jurisdiction, lower the score.
- Interpret Israeli time/currency/units correctly (ימים/שבועות/חודשים; ₪). Numeric expressions may appear as digits or words.
- Base your decision only on the provided text; use anchors to calibrate the 0–4 scale for this query.

Instructions
- Evaluate candidate.passage against the query using the scale, calibrated by the anchors.
- Return JSON only, one line, exactly in this format:
{"doc_id":"<candidate.doc_id>","label":0,"confidence":0.0}

Where label ∈ {0,1,2,3,4} and confidence ∈ [0,1].
"""

def dprint(*a):
    if DEBUG:
        print(*a, flush=True)

def robust_json(text: str) -> Dict[str, Any]:
    """Extract JSON from possibly fenced output."""
    try:
        return json.loads(text)
    except:
        pass
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.DOTALL)
    try:
        return json.loads(t)
    except:
        pass
    start, end = t.find("{"), t.rfind("}")
    if start >= 0 and end > start:
        return json.loads(t[start:end+1])
    raise ValueError("Could not parse JSON")

async def call_gpt_with_reasoning(client: AsyncOpenAI, payload: Dict) -> Dict:
    """Call GPT model using responses.create with reasoning enabled."""
    last_err = None
    for attempt in range(1, MAX_RETRY + 1):
        try:
            resp = await client.responses.create(
                model=MODEL_GPT,
                input=[
                    {"role": "system", "content": SYSTEM_PROMPT_GPT},
                    {"role": "user", "content": json.dumps(payload, ensure_ascii=False)}
                ],
                # Reasoning settings (as used in new.py)
                text={"format": {"type": "text"}, "verbosity": "low"},
                reasoning={"effort": "medium", "summary": "auto"},
                store=False,
                tools=[],
            )
            
            # Robust extraction of text from response
            text = getattr(resp, "output_text", None)
            if not text and hasattr(resp, "output") and isinstance(resp.output, list):
                parts = []
                for item in resp.output:
                    for c in getattr(item, "content", []) or []:
                        t = getattr(c, "text", None)
                        if isinstance(t, str):
                            parts.append(t)
                if parts:
                    text = "\n".join(parts)
            if not text and hasattr(resp, "content") and isinstance(resp.content, list):
                parts = []
                for c in resp.content:
                    t = getattr(c, "text", None)
                    if isinstance(t, str):
                        parts.append(t)
                if parts:
                    text = "\n".join(parts)
            
            dprint(f"[model raw#{attempt}] {(text or '')[:200]}{'…' if text and len(text) > 200 else ''}")
            
            if not text:
                raise RuntimeError("Empty response from model")
            
            out = robust_json(text)
            if "doc_id" not in out or "label" not in out:
                raise ValueError("Missing required keys in model output")
            lab = int(out["label"])
            if lab < 0 or lab > 4:
                raise ValueError("Label out of range")
            return {
                "doc_id": str(out.get("doc_id", "")),
                "label": lab,
                "confidence": float(out.get("confidence", 0.0))
            }
        except Exception as e:
            last_err = e
            dprint(f"[warn] attempt {attempt} failed: {e}")
            if attempt < MAX_RETRY:
                await asyncio.sleep(min(1.5 * attempt, 6.0))
            else:
                raise last_err

async def run_gpt_labeling(input_file: str, output_file: str):
    """Run GPT labeling on all packages using responses.create with reasoning."""
    if not OPENAI_API_KEY:
        print("[SKIP] OPENAI_API_KEY not set")
        return
    
    if not Path(input_file).exists():
        print(f"[SKIP] Input file not found: {input_file}")
        return
    
    # Resume: load already done pairs
    done = set()
    if Path(output_file).exists():
        for row in iter_jsonl(output_file):
            done.add((row.get("query_uuid"), row.get("doc_id")))
        print(f"[RESUME] {len(done)} already labeled")
    
    client = AsyncOpenAI(api_key=OPENAI_API_KEY)
    sem = asyncio.Semaphore(CONCURRENCY_GPT)
    lock = asyncio.Lock()
    
    out_fh = open(output_file, "a" if done else "w", encoding="utf-8")
    
    async def process(rec: Dict, idx: int):
        qid = rec["query"]["uuid"]
        doc_id = rec["candidate"]["doc_id"]
        if (qid, doc_id) in done:
            return
        
        async with sem:
            try:
                result = await call_gpt_with_reasoning(client, rec)
                row = {
                    "query_uuid": qid,
                    "doc_id": doc_id,
                    "label": result["label"],
                    "confidence": result["confidence"],
                    "model": MODEL_GPT
                }
                async with lock:
                    out_fh.write(json.dumps(row, ensure_ascii=False) + "\n")
                    out_fh.flush()
            except Exception as e:
                print(f"[ERR] q={qid} doc={doc_id}: {e}", file=sys.stderr)
    
    tasks = []
    for i, rec in enumerate(iter_jsonl(input_file), 1):
        tasks.append(asyncio.create_task(process(rec, i)))
    
    print(f"[GPT-5] Processing {len(tasks)} packages (model={MODEL_GPT}, concurrency={CONCURRENCY_GPT})...")
    await asyncio.gather(*tasks)
    out_fh.close()
    print(f"[GPT-5] Done -> {output_file}")

In [ ]:
# Run GPT-5 labeling for crit_ge4 detection
# Set OPENAI_API_KEY in environment before running

GPT_INPUT = os.path.join(OUTPUT_DIR, "llm_inputs_relabel_crit_ge4.jsonl")
GPT_OUTPUT = os.path.join(OUTPUT_DIR, "llm_labels_gpt5_ge4.jsonl")

# Uncomment to run:
# await run_gpt_labeling(input_file=GPT_INPUT, output_file=GPT_OUTPUT)

In [ ]:
# ==================== GEMINI LABELING ====================
# Aligned with final_relabel/new_gem.py

from google import genai
from google.genai import types

# Config
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")  # Set your key
MODEL_GEMINI = os.getenv("MODEL_GEMINI", "gemini-2.5-pro")
CONCURRENCY_GEMINI = int(os.getenv("CONCURRENCY_GEMINI", "6"))
THINKING_BUDGET = int(os.getenv("THINKING_BUDGET", "-1"))  # -1=dynamic, 0=off, N=tokens
INCLUDE_THOUGHTS = bool(int(os.getenv("INCLUDE_THOUGHTS", "0")))

SYSTEM_PROMPT_GEMINI = """Rate the relevance of a passage to a query on a 0–4 scale using a single JSON object you will receive.

Input JSON fields:
- query: { "uuid", "text" }
- anchors: list of examples from the same query, each { "label": 0..4, "passage": "…" } (use these to calibrate the scale)
- candidate: { "doc_id", "passage" } (the passage to score)
- ref4: { "doc_id", "passage" } (an additional reference passage from the same query)

Relevance scale (0–4)
0 – No substantive relation to the query.
1 – Weak/general relation; no direct answer.
2 – Partially relevant; some matching info but key elements are missing.
3 – Strong relevance; mostly answers the query with minor gaps.
4 – Fully relevant; a direct and complete answer.

Hebrew & Israel cues
- Texts are in Hebrew (RTL) and typically without niqqud; treat common Hebrew prefixes/suffixes (ב/כ/ל/מ/ו/ה-, -ים/-ות) and spelling variants as equivalent.
- Prefer content grounded in the Israeli context where relevant (e.g., laws/procedures/institutions such as ביטוח לאומי, שירות התעסוקה, רשות המסים, משרד הפנים, קופות חולים). If the passage is clearly about a non-Israeli jurisdiction for an Israeli query, lower the score.
- Interpret Israeli time/currency/units correctly (ימים/שבועות/חודשים; ₪). Numbers may appear as digits or words.

Instructions
- Evaluate candidate.passage against the query using the scale, calibrated by the anchors.
- Return JSON only, one line, exactly in this format:
{"doc_id":"<candidate.doc_id>","label":0,"confidence":0.0}

Where label ∈ {0,1,2,3,4} and confidence ∈ [0,1]."""

def build_gemini_client() -> genai.Client:
    return genai.Client(api_key=GEMINI_API_KEY)

def build_gemini_config() -> types.GenerateContentConfig:
    """Build config with thinking_config as per Gemini docs."""
    return types.GenerateContentConfig(
        system_instruction=types.Content(
            role="system",
            parts=[types.Part.from_text(text=SYSTEM_PROMPT_GEMINI)]
        ),
        thinking_config=types.ThinkingConfig(
            thinking_budget=THINKING_BUDGET,
            include_thoughts=INCLUDE_THOUGHTS
        ),
    )

def call_gemini_sync(client: genai.Client, model: str, payload: Dict[str, Any]) -> Dict[str, Any]:
    """Synchronous Gemini call that returns parsed JSON: {'doc_id','label','confidence'}."""
    cfg = build_gemini_config()
    contents = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=json.dumps(payload, ensure_ascii=False))]
        )
    ]
    
    resp = client.models.generate_content(model=model, contents=contents, config=cfg)
    
    # Extract text from response
    out_text = (getattr(resp, "text", None) or "").strip()
    
    # If empty, assemble from non-thought parts
    if not out_text and resp.candidates:
        parts = []
        cand = resp.candidates[0]
        if cand and cand.content and getattr(cand.content, "parts", None):
            for part in cand.content.parts:
                # Skip thought summaries
                if getattr(part, "thought", False):
                    continue
                if getattr(part, "text", None):
                    parts.append(part.text)
        out_text = "".join(parts).strip()
    
    dprint(f"[gemini raw] {out_text[:240]}{'…' if out_text and len(out_text) > 240 else ''}")
    
    if not out_text:
        raise RuntimeError("Empty response from Gemini")
    
    out = robust_json(out_text)
    if "doc_id" not in out or "label" not in out:
        raise ValueError("Missing required keys in model output")
    lab = int(out["label"])
    if lab < 0 or lab > 4:
        raise ValueError(f"Label out of range: {lab}")
    return {
        "doc_id": str(out.get("doc_id", "")),
        "label": lab,
        "confidence": float(out.get("confidence", 0.0))
    }

async def run_gemini_labeling(input_file: str, output_file: str):
    """Run Gemini labeling on all packages with thinking enabled."""
    if not GEMINI_API_KEY:
        print("[SKIP] GEMINI_API_KEY not set")
        return
    
    if not Path(input_file).exists():
        print(f"[SKIP] Input file not found: {input_file}")
        return
    
    # Resume: load already done pairs
    done = set()
    if Path(output_file).exists():
        for row in iter_jsonl(output_file):
            done.add((row.get("query_uuid"), row.get("doc_id")))
        print(f"[RESUME] {len(done)} already labeled")
    
    client = build_gemini_client()
    sem = asyncio.Semaphore(CONCURRENCY_GEMINI)
    lock = asyncio.Lock()
    
    out_fh = open(output_file, "a" if done else "w", encoding="utf-8")
    
    async def worker(rec: Dict[str, Any], idx: int):
        qid = rec["query"]["uuid"]
        doc_id = rec["candidate"]["doc_id"]
        if (qid, doc_id) in done:
            return
        
        async with sem:
            try:
                # Use asyncio.to_thread for sync Gemini call
                result = await asyncio.to_thread(call_gemini_sync, client, MODEL_GEMINI, rec)
                row = {
                    "query_uuid": qid,
                    "doc_id": doc_id,
                    "label": result["label"],
                    "confidence": result["confidence"],
                    "model": MODEL_GEMINI
                }
                async with lock:
                    out_fh.write(json.dumps(row, ensure_ascii=False) + "\n")
                    out_fh.flush()
            except Exception as e:
                print(f"[ERR] q={qid} doc={doc_id}: {e}", file=sys.stderr)
    
    tasks = []
    for i, rec in enumerate(iter_jsonl(input_file), 1):
        tasks.append(asyncio.create_task(worker(rec, i)))
    
    print(f"[GEMINI] Processing {len(tasks)} packages (model={MODEL_GEMINI}, thinking_budget={THINKING_BUDGET})...")
    await asyncio.gather(*tasks)
    out_fh.close()
    print(f"[GEMINI] Done -> {output_file}")

In [ ]:
# Run Gemini labeling for crit_ge4 detection
# Set GEMINI_API_KEY in environment before running

GEMINI_INPUT = os.path.join(OUTPUT_DIR, "llm_inputs_relabel_crit_ge4.jsonl")
GEMINI_OUTPUT = os.path.join(OUTPUT_DIR, "llm_labels_gemini_ge4.jsonl")

# Uncomment to run:
# await run_gemini_labeling(input_file=GEMINI_INPUT, output_file=GEMINI_OUTPUT)

## Step 4: Compare & Combine Labels

Compare GPT-5 and Gemini outputs, then combine using precision-first rules.

In [ ]:
# ==================== COMPARE LABELS ====================
# Adapted from final_relabel/compare.py

def load_label_map(path: str) -> Dict[Tuple[str, str], Dict]:
    """Load labels: (qid, doc_id) -> {label, confidence}."""
    m = {}
    if not Path(path).exists():
        return m
    for row in iter_jsonl(path):
        qid = row.get("query_uuid")
        doc_id = row.get("doc_id")
        if qid and doc_id:
            m[(qid, doc_id)] = {
                "label": int(row.get("label", 0)),
                "conf": float(row.get("confidence", 0.0))
            }
    return m

def compare_labels(gpt_path: str, gemini_path: str):
    """Compare agreement between two label files."""
    gpt = load_label_map(gpt_path)
    gemini = load_label_map(gemini_path)
    
    keys = set(gpt) & set(gemini)
    if not keys:
        print("No overlapping pairs.")
        return
    
    agree = sum(1 for k in keys if gpt[k]["label"] == gemini[k]["label"])
    print(f"Overlapping pairs: {len(keys)}")
    print(f"Agreement: {agree}/{len(keys)} ({100*agree/len(keys):.1f}%)")
    
    # Confusion matrix
    cm = [[0]*5 for _ in range(5)]
    for k in keys:
        g, h = gpt[k]["label"], gemini[k]["label"]
        if 0 <= g <= 4 and 0 <= h <= 4:
            cm[g][h] += 1
    
    print("\nConfusion Matrix (rows=GPT, cols=Gemini):")
    print("     " + "  ".join(f"{i:3d}" for i in range(5)))
    for i in range(5):
        print(f"{i}:   " + "  ".join(f"{cm[i][j]:3d}" for j in range(5)))

In [ ]:
# Compare labels (uncomment when both files exist)
compare_labels(
    gpt_path=os.path.join(OUTPUT_DIR, "llm_labels_gpt5_ge4.jsonl"),
    gemini_path=os.path.join(OUTPUT_DIR, "llm_labels_gemini_ge4.jsonl")
)

In [ ]:
# ==================== COMBINE LABELS ====================
# Adapted from final_relabel/combine.py

POS_CONF_MIN = float(os.getenv("POS_CONF_MIN", "0.80"))  # for label 3/4
NEG_CONF_MIN = float(os.getenv("NEG_CONF_MIN", "0.60"))  # for label 0

def combine_labels(gpt_path: str, gemini_path: str, output_path: str):
    """
    Combine labels using precision-first rules:
    - Rule 4a: GPT confident label=4
    - Rule 4b: Both agree on 4 with high confidence
    - Rule 3: Both >= 3 with GPT confident
    - Rule 0: Both <= 1 with reasonable confidence
    """
    gpt = load_label_map(gpt_path)
    gemini = load_label_map(gemini_path)
    
    keys = sorted(set(gpt) & set(gemini))
    if not keys:
        print("No overlapping pairs.")
        return
    
    kept = []
    stats = defaultdict(int)
    
    for qid, doc_id in keys:
        g = gpt[(qid, doc_id)]
        h = gemini[(qid, doc_id)]
        gL, gC = g["label"], g["conf"]
        hL, hC = h["label"], h["conf"]
        minC = min(gC, hC)
        
        # Rule 4a: GPT confident 4
        if gL == 4 and gC >= POS_CONF_MIN:
            kept.append({"query_uuid": qid, "doc_id": doc_id, "label": 4, "source": "rule_4a"})
            stats["rule_4a"] += 1
            continue
        
        # Rule 4b: Both 4, both confident
        if gL == 4 and hL == 4 and minC >= POS_CONF_MIN:
            kept.append({"query_uuid": qid, "doc_id": doc_id, "label": 4, "source": "rule_4b"})
            stats["rule_4b"] += 1
            continue
        
        # Rule 3: Both >= 3, GPT confident
        if gL >= 3 and hL >= 3 and gC >= POS_CONF_MIN:
            kept.append({"query_uuid": qid, "doc_id": doc_id, "label": 3, "source": "rule_3"})
            stats["rule_3"] += 1
            continue
        
        # Rule 0: Both <= 1, reasonably confident
        if gL <= 1 and hL <= 1 and minC >= NEG_CONF_MIN:
            kept.append({"query_uuid": qid, "doc_id": doc_id, "label": 0, "source": "rule_0"})
            stats["rule_0"] += 1
            continue
        
        stats["skipped"] += 1
    
    with open(output_path, "w", encoding="utf-8") as f:
        for r in kept:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    
    print(f"Input pairs: {len(keys)}")
    print(f"Output: {output_path}")
    print(f"\nCounts:")
    for k in ["rule_4a", "rule_4b", "rule_3", "rule_0", "skipped"]:
        print(f"  {k}: {stats.get(k, 0)}")
    print(f"\nKept: {len(kept)} | Skipped: {stats.get('skipped', 0)}")

In [ ]:
# Combine labels (uncomment when both files exist)
combine_labels(
    gpt_path=os.path.join(OUTPUT_DIR, "llm_labels_gpt5_ge4.jsonl"),
    gemini_path=os.path.join(OUTPUT_DIR, "llm_labels_gemini_ge4.jsonl"),
    output_path=os.path.join(OUTPUT_DIR, "llm_labels_combined_ge4.jsonl")
)

## Step 5: Merge Back to Training Data

Merge the combined labels back into the training dataset.

In [ ]:
# ==================== MERGE TO TRAINING ====================

def merge_relabels_to_train(
    combined_labels_path: str,
    train_jsonl_path: str,
    output_train_path: str
):
    """
    Merge relabeled passages back into training data.
    Updates labels for passages that were relabeled.
    """
    # Load relabels
    relabels = {}  # (qid, doc_id) -> label
    for row in iter_jsonl(combined_labels_path):
        qid = row.get("query_uuid")
        doc_id = row.get("doc_id")
        label = row.get("label")
        if qid and doc_id and label is not None:
            relabels[(qid, doc_id)] = int(label)
    
    print(f"[MERGE] Loaded {len(relabels)} relabels")
    
    # Process training data
    updated = 0
    with open(output_train_path, "w", encoding="utf-8") as wf:
        for row in iter_jsonl(train_jsonl_path):
            qid = row.get("query_uuid")
            paragraphs = row.get("paragraphs", {})
            target_actions = row.get("target_actions", {})
            
            # Check each paragraph
            for key, para_info in paragraphs.items():
                idx = key.replace("paragraph_", "")
                target_key = f"target_action_{idx}"
                doc_id = para_info.get("uuid") or para_info.get("paragraph_uuid")
                
                # Check if we have a relabel
                if doc_id and (qid, doc_id) in relabels:
                    new_label = relabels[(qid, doc_id)]
                    old_label = target_actions.get(target_key)
                    if old_label is None or old_label != new_label:
                        target_actions[target_key] = new_label
                        updated += 1
            
            row["target_actions"] = target_actions
            wf.write(json.dumps(row, ensure_ascii=False) + "\n")
    
    print(f"[MERGE] Updated {updated} labels -> {output_train_path}")

In [ ]:
# Merge relabels to training (uncomment to run)
# merge_relabels_to_train(
#     combined_labels_path=os.path.join(OUTPUT_DIR, "llm_labels_combined_ge4.jsonl"),
#     train_jsonl_path=TRAIN_JSONL,
#     output_train_path=os.path.join(OUTPUT_DIR, "hsrc_train_relabeled.jsonl")
# )

## Summary

This notebook implements the complete Critical Passage Relabeling pipeline:

1. **Load Data** - Training data, corpus, fused rankings
2. **Build LLM Input Packages** - Creates packages with same-query anchors
3. **LLM Labeling** - GPT-5 and Gemini labeling (async)
4. **Compare & Combine** - Precision-first combination of both models
5. **Merge to Training** - Update training data with new labels

### How Anchors Work

**Anchors = One passage per label (0-4) from the same query**

For each query, we select up to 5 calibration passages:
- Label 0: "No relevance" example
- Label 1: "Weak relevance" example  
- Label 2: "Partial relevance" example
- Label 3: "Strong relevance" example
- Label 4: "Perfect relevance" example

When multiple passages have the same label, we pick the one with best fused rank.

### How ref4 Works

**ref4 = A label-4 passage ranked BELOW the candidate**

This gives the LLM a "gold standard" comparison:
- Priority: Label-4 passage that ranks lower than the candidate
- Fallback: Best-ranked label-4 from this query

If the candidate is truly relevant, it should be comparable to ref4.